Kenya EV Market Analysis

Phase 1: EV Market Adoption & Category Trends

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_adoption = pd.read_csv('kenya_ev_adoption.csv')


print("--- First 5 Rows of Data ---")
print(df_adoption.head())

Aggregate the Data (The Pivot Table)

In [ ]:
df_trends = df_adoption.groupby(['Year', 'EV_Type'])['Registrations'].sum().unstack().fillna(0)

print("\n--- Annual Registrations Matrix ---")
print(df_trends)

Total Market Share & YoY Growth

In [ ]:
df_total_by_type = df_adoption.groupby('EV_Type')['Registrations'].sum().sort_values(ascending=False)
print("\n--- Cumulative Registrations (2018-2024) ---")
print(df_total_by_type)


df_annual_total = df_adoption.groupby('Year')['Registrations'].sum()
growth_rates = df_annual_total.pct_change() * 100
print("\n--- Market YoY Growth Rates (%) ---")
print(growth_rates.round(2))

Visual Chart

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")


df_trends.plot(kind='area', stacked=True, alpha=0.85, ax=plt.gca(), cmap='viridis')


plt.title('Kenya EV Adoption Trends by Vehicle Category (2018-2024)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total Registrations', fontsize=12)
plt.legend(title='Vehicle Type', loc='upper left')
plt.tight_layout()


plt.savefig('my_kenya_ev_trends.png', dpi=300)


plt.show()

The Boda Boda Phenomenon: Note that Motorcycles distort the axis scale because they represent over 75% of the total volume.
The 2024 Bounce: Point out how growth surged again in 2024 after a flat 2023, reflecting real-world implementation of favorable e-mobility electricity tariffs and policy updates in Kenya.

Evaluating Localized Infrastructure Growth

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df_infra = pd.read_csv('kenya_charging_infra.csv')
df_adoption = pd.read_csv('kenya_ev_adoption.csv')


print("--- Charger Type Distribution ---")
print(df_infra['Charger_Type'].value_counts())

print("\n--- Top Infrastructure Operators ---")
print(df_infra['Operator'].value_counts())


df_2024 = df_adoption[df_adoption['Year'] == 2024]
county_evs = df_2024.groupby('County')['Registrations'].sum()


operational_stations = df_infra[df_infra['Status'] == 'Operational'].groupby('County').size()


ratio_df = pd.DataFrame({
    '2024_EV_Registrations': county_evs,
    'Operational_Stations': operational_stations
}).fillna(0)

ratio_df['EV_per_Station_Ratio'] = (ratio_df['2024_EV_Registrations'] / ratio_df['Operational_Stations']).round(1)
ratio_df_sorted = ratio_df.reset_index().sort_values(by='EV_per_Station_Ratio', ascending=False)

print("\n--- EV-to-Station Ratio Table ---")
print(ratio_df_sorted.to_string(index=False))

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.countplot(data=df_infra, x='Charger_Type', ax=axes[0], palette='viridis', 
              order=df_infra['Charger_Type'].value_counts().index)
axes[0].set_title('Distribution of Charger Types in Kenya', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Charger Type')
axes[0].set_ylabel('Number of Stations')

sns.barplot(data=ratio_df_sorted, x='EV_per_Station_Ratio', y='County', ax=axes[1], palette='magma')
axes[1].set_title('EV-to-Operational Station Ratio by County (2024)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('EVs per Active Charging Station')
axes[1].set_ylabel('County')

plt.tight_layout()
plt.savefig('my_kenya_ev_infrastructure_analysis.png', dpi=300)
plt.show()

"While Nairobi is the natural starting point for e-mobility deployment, the data reveals critical infrastructure deficits in secondary transit hubs. Nakuru possesses an EV-to-station ratio nearly double that of Nairobi, signaling an underserved market. Furthermore, the high density of Battery Swap Stations relative to DC Fast Chargers underscores an industry heavily optimized for immediate, commercial two-wheeler turnaround rather than private passenger vehicles."

Energy Fuel vs. EV Charging Unit Economics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df_costs = pd.read_csv('kenya_fuel_vs_ev_costs.csv')

df_costs['Monthly_Fuel_Cost_KES'] = df_costs['Avg_Daily_Fuel_Cost_KES'] * 30
df_costs['Monthly_EV_Cost_KES'] = df_costs['Avg_Daily_EV_Cost_KES'] * 30
df_costs['Daily_Savings_KES'] = df_costs['Avg_Daily_Fuel_Cost_KES'] - df_costs['Avg_Daily_EV_Cost_KES']
df_costs['Monthly_Savings_KES'] = df_costs['Monthly_Fuel_Cost_KES'] - df_costs['Monthly_EV_Cost_KES']
df_costs['Savings_Percentage'] = (df_costs['Daily_Savings_KES'] / df_costs['Avg_Daily_Fuel_Cost_KES']) * 100

print("--- Operational Savings Calculations ---")
print(df_costs[['Month', 'Petrol_Price_KES_Ltr', 'Electricity_Tariff_KES_kWh', 'Daily_Savings_KES', 'Monthly_Savings_KES', 'Savings_Percentage']].to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_costs['Month'], df_costs['Avg_Daily_Fuel_Cost_KES'], marker='o', color='#d9534f', linewidth=2.5, label='Daily Petrol Cost')
ax.plot(df_costs['Month'], df_costs['Avg_Daily_EV_Cost_KES'], marker='s', color='#5cb85c', linewidth=2.5, label='Daily EV Charging Cost')

ax.set_title('Economic Incentive: Daily Petrol vs. EV Charging Costs (KES)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month (2024)', fontsize=12)
ax.set_ylabel('Daily Cost (KES)', fontsize=12)
ax.set_ylim(0, 1200)
ax.legend(loc='upper right')

for i, txt in enumerate(df_costs['Daily_Savings_KES']):
    ax.annotate(f"Save KES {txt}", (df_costs['Month'][i], df_costs['Avg_Daily_EV_Cost_KES'][i] + 80),
                 ha='center', fontsize=9, fontweight='bold', color='#2b542c')

plt.tight_layout()
plt.savefig('my_kenya_ev_cost_benefit_analysis.png', dpi=300)
plt.show()

The unit economics explain why adoption is accelerating despite high upfront vehicle acquisition costs. EV operations yield an approximate 79% reduction in daily energy costs compared to fossil fuels. Even when petrol prices softened slightly between January and May, the savings delta remained remarkably stable because electricity tariffs tracked downward simultaneously. For a commercial transit provider, an extra KES 23,700+ in monthly net margin dramatically shifts the asset payback period from years to months.

Macro Market Forecasting to 2030

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_forecast = pd.read_csv('kenya_ev_forecast.csv')

df_forecast['Annual_New_EV_Additions'] = df_forecast['Projected_Total_EVs'].diff().fillna(df_forecast['Projected_Total_EVs'].iloc[0] - 5000)
df_forecast['Electricity_Demand_YoY_Increase_MWh'] = df_forecast['Electricity_Demand_Forecast_MWh'].diff().fillna(df_forecast['Electricity_Demand_Forecast_MWh'].iloc[0])

print("--- EV Market Projections & Energy Demands (2025-2030) ---")
print(df_forecast[['Year', 'Projected_Total_EVs', 'Annual_New_EV_Additions', 'Electricity_Demand_Forecast_MWh', 'Stations_Needed']].to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(11, 6))


color = '#1f77b4'
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Total Projected EV Fleet', color=color, fontsize=12)
ax1.bar(df_forecast['Year'], df_forecast['Projected_Total_EVs'], color=color, alpha=0.6, label='Projected EV Fleet')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = '#ff7f0e'
ax2.set_ylabel('Annual Grid Demand (MWh)', color=color, fontsize=12)
ax2.plot(df_forecast['Year'], df_forecast['Electricity_Demand_Forecast_MWh'], color=color, marker='D', linewidth=2.5, label='Grid Demand (MWh)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Kenya EV Market Growth Strategy & Grid Impact (2025-2030)', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()

plt.savefig('my_kenya_ev_2030_forecast.png', dpi=300)
plt.show()

The trajectory from 2025 to 2030 highlights a classic compounding s-curve pattern typical of technology adoption lifecycles. For macro-investors, the critical value driver isn't just the vehicles themselves, but the ancillary infrastructure required to sustain them. To support over 50,000 EVs on Kenyan roads, private and public operators must deploy an additional 2,456 stations over the next 4 years.

Furthermore, with annual energy consumption crossing 107,000 MWh by 2030, the e-mobility sector shifts from an experimental pilot to a highly lucrative, predictable source of baseload demand for utility providers like Kenya Power—particularly beneficial given the nation's high composition of renewable geothermal and hydro energy sources.

Hardware Specifications & Range Efficiency Mapping

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_specs = pd.read_csv('kenya_ev_specs.csv')

df_specs['Efficiency_km_per_kWh'] = (df_specs['Range_km'] / df_specs['Battery_Capacity_kWh']).round(2)

print("--- Technical Performance Benchmarking ---")
print(df_specs[['Model', 'Category', 'Battery_Capacity_kWh', 'Range_km', 'Efficiency_km_per_kWh', 'Charging_Time_Mins']].to_string(index=False))

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_specs, 
    x='Battery_Capacity_kWh', 
    y='Range_km', 
    hue='Category', 
    style='Category',
    s=200, 
    palette='Set2'
)

for i in range(df_specs.shape[0]):
    plt.text(
        x=df_specs['Battery_Capacity_kWh'].iloc[i] + 2, 
        y=df_specs['Range_km'].iloc[i] - 5, 
        s=df_specs['Model'].iloc[i], 
        fontweight='bold', 
        fontsize=10
    )

plt.title('Technical Benchmarking: Battery Capacity vs. Operational Range', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Battery Capacity (kWh)', fontsize=12)
plt.ylabel('Stated Range (km)', fontsize=12)
plt.xlim(-5, 140)
plt.ylim(50, 460)
plt.tight_layout()


plt.savefig('my_kenya_ev_specs_benchmarking.png', dpi=300)
plt.show()

The engineering metrics show why the market has scaled the way it has. E-motorcycles deliver an energy efficiency profile that is 400% higher than passenger electric cars, combined with near-instantaneous refueling times through battery swapping modules. This completely eliminates the two biggest barriers to EV adoption: range anxiety and charging downtime for commercial riders.

For fleet management companies and logistics providers operating in East Africa, optimizing asset utilization means matching the right vehicle type to localized grid constraints. While passenger cars and delivery vans require intensive capital deployments for localized DC Fast Chargers, the two-wheeler ecosystem can rapidly expand via decentralized, low-voltage battery swap cabinets, making it the most capital-efficient pathway for near-term regional market growth.